In [1]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd

from utils.display_tools import load_best_forecasts


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
lot = pickle.load(open("../saves_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../saves_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../saves_and_results/cache_exogs.p", "rb"))

In [4]:
d = data["Moehne"]["desc1"]
y_ = d.loc[d.index.year == 2020]

### Get feature importance

In [5]:
chronos = pd.read_csv(
    "results/journal/chronos/Moehne_desc1_chronos_zero_shot_chronos.csv", index_col=0)
tfm = pd.read_csv(
    "results/journal/tfm/Moehne_desc1_tfm_zero_shot_chronos.csv", index_col=0)

model_spec, forecasts = load_best_forecasts(
        "results/journal/" + "0", direction="desc1"
    )
score = pd.DataFrame([np.abs(forecasts[x] - y_.values).mean() for x in forecasts.keys()], index = forecasts.keys())
score.loc["chronos"] = np.abs(y_ - chronos.values[:-1]).mean()
score.loc["tfm"] = np.abs(y_ - tfm.values[:-1]).mean()

In [6]:
best_runs = [score.index[score[x].argmin()] for x in score.columns]

In [7]:
indices = forecasts["sarimax"].columns

In [20]:
line_stack = []
for n,ind in enumerate(indices):

    a = model_spec.loc[(model_spec["Index"] == int(ind)) & (model_spec.index == best_runs[n])]
    line_stack.append(a)

In [21]:
specifics = pd.concat(line_stack)[['STAU', 'T', 'M_Stau', 'M_T', 'Interaction', "Decompose"]].T
specifics.columns = indices[2:]
specifics[indices[0]] = [[],[], 0,0,0,0]
specifics[indices[1]] = [[],[], 0,0,0,0]
specifics = specifics[indices]
specifics.loc["p_stau"] =specifics.loc["STAU"]
specifics.loc["p_T"] =specifics.loc["T"]

In [22]:
specifics.loc["STAU"] = specifics.loc["STAU"].str.contains("0")
specifics.loc["T"] = specifics.loc["T"].str.contains("0")
specifics.loc["M_Stau"] = specifics.loc["M_Stau"]> 0 
specifics.loc["M_T"] = specifics.loc["M_T"] >0
specifics.loc["Interaction"] = specifics.loc["Interaction"]> 0 
specifics.loc["p_stau"] = specifics.loc["p_stau"].str.contains("1")
specifics.loc["p_T"] = specifics.loc["p_T"].str.contains("1")
specifics.loc["Decompose"] = specifics.loc["Decompose"]> 0 

In [24]:
specifics[specifics.isnull()] = False

In [25]:
specifics = specifics.loc[["STAU", "T","p_stau", "p_T", "M_Stau", "M_T", "Interaction", "Decompose"]]

In [26]:
used = specifics.values.astype(bool)
specifics[used] = "cmark"
specifics[~used] = "xmark"

In [27]:
specifics

,10905,10961,10982,11038,11161,11176,11208,11243
STAU,xmark,xmark,cmark,cmark,xmark,xmark,cmark,cmark
T,xmark,xmark,xmark,cmark,cmark,cmark,cmark,cmark
p_stau,xmark,xmark,xmark,xmark,xmark,xmark,cmark,cmark
p_T,xmark,xmark,xmark,cmark,cmark,xmark,cmark,cmark
M_Stau,xmark,xmark,cmark,xmark,xmark,xmark,xmark,xmark
M_T,xmark,xmark,cmark,cmark,xmark,cmark,xmark,cmark
Interaction,xmark,xmark,cmark,cmark,cmark,xmark,cmark,xmark
Decompose,xmark,xmark,cmark,cmark,cmark,cmark,cmark,cmark


In [28]:
print(specifics.to_latex())

\begin{tabular}{lllllllll}
\toprule
{} &  10905 &  10961 &  10982 &  11038 &  11161 &  11176 &  11208 &  11243 \\
\midrule
STAU        &  xmark &  xmark &  cmark &  cmark &  xmark &  xmark &  cmark &  cmark \\
T           &  xmark &  xmark &  xmark &  cmark &  cmark &  cmark &  cmark &  cmark \\
p\_stau      &  xmark &  xmark &  xmark &  xmark &  xmark &  xmark &  cmark &  cmark \\
p\_T         &  xmark &  xmark &  xmark &  cmark &  cmark &  xmark &  cmark &  cmark \\
M\_Stau      &  xmark &  xmark &  cmark &  xmark &  xmark &  xmark &  xmark &  xmark \\
M\_T         &  xmark &  xmark &  cmark &  cmark &  xmark &  cmark &  xmark &  cmark \\
Interaction &  xmark &  xmark &  cmark &  cmark &  cmark &  xmark &  cmark &  xmark \\
Decompose   &  xmark &  xmark &  cmark &  cmark &  cmark &  cmark &  cmark &  cmark \\
\bottomrule
\end{tabular}



/tmp/ipykernel_1211290/2314209861.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(specifics.to_latex())
